In [2]:
import logging

logging.basicConfig(
  level = logging.INFO,
  datefmt="%M:%S",
  format="[%(asctime)s.%(msecs)03d] %(lineno)3d %(funcName)15s %(levelname)-8s = %(message)s",
  force=True,
)

log = logging.getLogger(__name__)

In [3]:
log.info("Hello")

[55:42.303]   1        <module> INFO     = Hello


In [4]:
import asyncio

async def producer(n_tasks: int):
  log.info("start producing")

  tasks = []
  for i in range(1, n_tasks+1):
    await asyncio.sleep(0.1)
    task = {'id': i, 'name' : f"task_{i:03}"}
    log.info("Prepare item %s", task)
    tasks.append(task)

  log.info("done producing")
  return tasks

async def process_task(task: dict):
  log.info("process task %s", task)
  await asyncio.sleep(0.1)
  log.info("done task %s", task)

async def consumer(tasks: list[dict]):
  log.info("start consuming")
  for task in tasks:
    await process_task(task)
  log.info("done consuming")





In [5]:
async def main():
  tasks = await producer(10)
  await consumer(tasks)

await main()

[55:42.328]   4        producer INFO     = start producing
[55:42.439]  10        producer INFO     = Prepare item {'id': 1, 'name': 'task_001'}
[55:42.549]  10        producer INFO     = Prepare item {'id': 2, 'name': 'task_002'}
[55:42.659]  10        producer INFO     = Prepare item {'id': 3, 'name': 'task_003'}
[55:42.768]  10        producer INFO     = Prepare item {'id': 4, 'name': 'task_004'}
[55:42.877]  10        producer INFO     = Prepare item {'id': 5, 'name': 'task_005'}
[55:42.987]  10        producer INFO     = Prepare item {'id': 6, 'name': 'task_006'}
[55:43.096]  10        producer INFO     = Prepare item {'id': 7, 'name': 'task_007'}
[55:43.205]  10        producer INFO     = Prepare item {'id': 8, 'name': 'task_008'}
[55:43.315]  10        producer INFO     = Prepare item {'id': 9, 'name': 'task_009'}
[55:43.423]  10        producer INFO     = Prepare item {'id': 10, 'name': 'task_010'}
[55:43.424]  13        producer INFO     = done producing
[55:43.424]  22       

In [6]:

async def producer(queue: asyncio.Queue, n_tasks: int):
  log.info("start producing")

  tasks = []
  for i in range(1, n_tasks+1):
    await asyncio.sleep(0.1)
    task = {'id': i, 'name' : f"task_{i:03}"}
    log.info("Prepare item %s", task)
    await queue.put(task)

  log.info("done producing")

async def consumer(queue: asyncio.Queue):
  log.info("start consuming")
  while not queue.empty():
    task = await queue.get()
    await process_task(task)
  log.info("done consuming")


In [7]:
async def main():  # blocking each others
  queue = asyncio.Queue()
  await producer(queue, n_tasks=10)
  await consumer(queue)

await main()

[55:44.541]   2        producer INFO     = start producing
[55:44.644]   8        producer INFO     = Prepare item {'id': 1, 'name': 'task_001'}
[55:44.754]   8        producer INFO     = Prepare item {'id': 2, 'name': 'task_002'}
[55:44.864]   8        producer INFO     = Prepare item {'id': 3, 'name': 'task_003'}
[55:44.973]   8        producer INFO     = Prepare item {'id': 4, 'name': 'task_004'}
[55:45.083]   8        producer INFO     = Prepare item {'id': 5, 'name': 'task_005'}
[55:45.193]   8        producer INFO     = Prepare item {'id': 6, 'name': 'task_006'}
[55:45.302]   8        producer INFO     = Prepare item {'id': 7, 'name': 'task_007'}
[55:45.411]   8        producer INFO     = Prepare item {'id': 8, 'name': 'task_008'}
[55:45.520]   8        producer INFO     = Prepare item {'id': 9, 'name': 'task_009'}
[55:45.630]   8        producer INFO     = Prepare item {'id': 10, 'name': 'task_010'}
[55:45.631]  11        producer INFO     = done producing
[55:45.632]  14       

In [8]:
async def main():  # blocking each others
  queue = asyncio.Queue()
  async with asyncio.TaskGroup() as tg:
    tg.create_task(producer(queue, n_tasks=10))
    tg.create_task(consumer(queue))

  await producer(queue, n_tasks=10)
  await consumer(queue)

await main()

[55:46.736]   2        producer INFO     = start producing
[55:46.737]  14        consumer INFO     = start consuming
[55:46.737]  18        consumer INFO     = done consuming
[55:46.848]   8        producer INFO     = Prepare item {'id': 1, 'name': 'task_001'}
[55:46.958]   8        producer INFO     = Prepare item {'id': 2, 'name': 'task_002'}
[55:47.068]   8        producer INFO     = Prepare item {'id': 3, 'name': 'task_003'}
[55:47.178]   8        producer INFO     = Prepare item {'id': 4, 'name': 'task_004'}
[55:47.286]   8        producer INFO     = Prepare item {'id': 5, 'name': 'task_005'}
[55:47.395]   8        producer INFO     = Prepare item {'id': 6, 'name': 'task_006'}
[55:47.504]   8        producer INFO     = Prepare item {'id': 7, 'name': 'task_007'}
[55:47.614]   8        producer INFO     = Prepare item {'id': 8, 'name': 'task_008'}
[55:47.722]   8        producer INFO     = Prepare item {'id': 9, 'name': 'task_009'}
[55:47.832]   8        producer INFO     = Prepare

In [9]:


async def consumer(queue: asyncio.Queue):
  log.info("init consumer")
  await asyncio.sleep(0.1)
  log.info("start consuming")
  while not queue.empty():
    task = await queue.get()
    await process_task(task)
  log.info("done consuming")


async def main():  # blocking each others
  queue = asyncio.Queue()
  async with asyncio.TaskGroup() as tg:
    tg.create_task(producer(queue, n_tasks=10))
    tg.create_task(consumer(queue))

await main()


[55:51.126]   2        producer INFO     = start producing
[55:51.127]   2        consumer INFO     = init consumer
[55:51.240]   8        producer INFO     = Prepare item {'id': 1, 'name': 'task_001'}
[55:51.241]   4        consumer INFO     = start consuming
[55:51.242]  17    process_task INFO     = process task {'id': 1, 'name': 'task_001'}
[55:51.350]   8        producer INFO     = Prepare item {'id': 2, 'name': 'task_002'}
[55:51.350]  19    process_task INFO     = done task {'id': 1, 'name': 'task_001'}
[55:51.351]  17    process_task INFO     = process task {'id': 2, 'name': 'task_002'}
[55:51.458]   8        producer INFO     = Prepare item {'id': 3, 'name': 'task_003'}
[55:51.459]  19    process_task INFO     = done task {'id': 2, 'name': 'task_002'}
[55:51.459]  17    process_task INFO     = process task {'id': 3, 'name': 'task_003'}
[55:51.568]   8        producer INFO     = Prepare item {'id': 4, 'name': 'task_004'}
[55:51.568]  19    process_task INFO     = done task {'id

In [10]:
async def process_task(task: dict):
  log.info("process task %s", task)
  await asyncio.sleep(0.09)
  log.info("done task %s", task)

In [11]:
async def main():  # blocking each others
  queue = asyncio.Queue()
  async with asyncio.TaskGroup() as tg:
    tg.create_task(producer(queue, n_tasks=10))
    tg.create_task(consumer(queue))

await main()


[55:52.356]   2        producer INFO     = start producing
[55:52.357]   2        consumer INFO     = init consumer
[55:52.460]   8        producer INFO     = Prepare item {'id': 1, 'name': 'task_001'}
[55:52.461]   4        consumer INFO     = start consuming
[55:52.462]   2    process_task INFO     = process task {'id': 1, 'name': 'task_001'}
[55:52.555]   4    process_task INFO     = done task {'id': 1, 'name': 'task_001'}
[55:52.555]   8        consumer INFO     = done consuming
[55:52.571]   8        producer INFO     = Prepare item {'id': 2, 'name': 'task_002'}
[55:52.680]   8        producer INFO     = Prepare item {'id': 3, 'name': 'task_003'}
[55:52.789]   8        producer INFO     = Prepare item {'id': 4, 'name': 'task_004'}
[55:52.899]   8        producer INFO     = Prepare item {'id': 5, 'name': 'task_005'}
[55:53.009]   8        producer INFO     = Prepare item {'id': 6, 'name': 'task_006'}
[55:53.119]   8        producer INFO     = Prepare item {'id': 7, 'name': 'task_00

In [12]:
import asyncio
import logging

logging.basicConfig(
  level = logging.INFO,
  datefmt="%M:%S",
  format="[%(asctime)s.%(msecs)03d] %(lineno)3d %(funcName)15s %(levelname)-8s = %(message)s",
  force=True,
)
log = logging.getLogger(__name__)

stop = object()

async def producer(queue: asyncio.Queue, n_tasks: int):
  log.info("start producing")
  for i in range(1, n_tasks + 1):
    await asyncio.sleep(0.1)
    task = {'id': i, 'name': f"task_{i:03}"}
    log.info("Prepare item %s", task)
    await queue.put(task)

  await queue.put(stop)
  log.info("done producing")

async def process_task(task: dict):
  log.info("process task %s", task)
  await asyncio.sleep(0.1)
  log.info("done task %s", task)

async def consumer(queue: asyncio.Queue):
  log.info("start consuming")
  while (task := await queue.get()) is not stop:
    await process_task(task)

  log.info("done consuming")


In [13]:
async def main():
  queue = asyncio.Queue()
  async with asyncio.TaskGroup() as tg:
    tg.create_task(producer(queue, n_tasks=10))
    tg.create_task(consumer(queue))


await main()


[55:53.473]  15        producer INFO     = start producing
[55:53.474]  31        consumer INFO     = start consuming
[55:53.587]  19        producer INFO     = Prepare item {'id': 1, 'name': 'task_001'}
[55:53.588]  26    process_task INFO     = process task {'id': 1, 'name': 'task_001'}
[55:53.696]  19        producer INFO     = Prepare item {'id': 2, 'name': 'task_002'}
[55:53.696]  28    process_task INFO     = done task {'id': 1, 'name': 'task_001'}
[55:53.697]  26    process_task INFO     = process task {'id': 2, 'name': 'task_002'}
[55:53.804]  19        producer INFO     = Prepare item {'id': 3, 'name': 'task_003'}
[55:53.805]  28    process_task INFO     = done task {'id': 2, 'name': 'task_002'}
[55:53.806]  26    process_task INFO     = process task {'id': 3, 'name': 'task_003'}
[55:53.914]  19        producer INFO     = Prepare item {'id': 4, 'name': 'task_004'}
[55:53.915]  28    process_task INFO     = done task {'id': 3, 'name': 'task_003'}
[55:53.915]  26    process_task

In [14]:
async def process_task(task: dict):
  log.info("process task %s", task)
  await asyncio.sleep(0.15)
  log.info("done task %s", task)